# Dispersion Trade — Guided Tutorial

This notebook implements a **dispersion trade** strategy on the **Nasdaq-100 (NDX)** index using index-tracking optimization constrained by implied volatility (IV) data from options markets.

---

## Overview

### What is a Dispersion Trade?

A **dispersion trade** exploits the structural gap between:

| Quantity | Description |
|---|---|
| $\sigma_{\text{index}}$ | Implied volatility of the *index* as a whole |
| $\bar{\sigma}_i$ | Weighted average of *single-stock* IVs |

Empirically, $\sigma_{\text{index}} < \bar{\sigma}_i$ because diversification reduces portfolio variance. The spread between the two represents a **correlation premium** that traders can capture.


---

**Follow the cells below step by step.** Each section explains the math and guides you through the inputs.

In [1]:
import os
import glob
import pandas as pd
import numpy as np

import iq.api.iqrestapi
import iq.tools.quadratic_utility
import iq.finance.index_tracking

# Initialize API credentials
iq.api.iqrestapi.initialize_credentials("7ec5a9ab48f24657bf3d6864da43b9be")


---

## Step 1 — Configuration

Set the parameters that define your dispersion trade. The key inputs are:

| Parameter | Type | Description |
|---|---|---|
| `DATA_DIR` | `str` | Root directory for the data package provided by IQ |
| `date_from` / `date_to` | `str` | Date range for the analysis (`YYYY-MM-DD`) |
| `V_0` | `float` | Upper bound on the IV ratio $\sigma_i / \sigma_{\text{index}}$ |
| `K` | `int` | Target number of stocks in the portfolio |

### The IV Ratio Bound $V_0$

$V_0$ is the core dispersion-trade lever. It controls how aggressively you can tilt toward high-IV stocks:

- **Higher $V_0$** → more flexible portfolio, potentially higher IV premium captured
- **Lower $V_0$** → tighter IV constraint, closer to pure index replication

Typical values: $V_0 \in [1.2,\ 2.0]$.

> **Data range:** The supplied dataset covers `2023-01-03` to `2025-12-15`. Any dates outside this window will trigger a validation error below.

In [2]:

DATA_DIR         = "data/dispersion_trade" #To be set by the user pointing to the specific data and structure sent by us.
STOCK_PRICES_DIR = os.path.join(DATA_DIR, "index_stock_data", "filtered_files")
NDX_PRICE_FILE   = os.path.join(DATA_DIR, "index_stock_data", "NDX_full_1day.txt")
OPTIONS_DATA_DIR = os.path.join(DATA_DIR, "options_data", "final_files")
NDX_IV_FILE      = os.path.join(DATA_DIR, "options_data", "NDX_filtered.csv")

BENCHMARK_COLUMN = "NDX Index"
date_from        = "2023-02-03"  # in this example it cant be prior to 2023-01-03 because of the options data availability
date_to          = "2025-10-15"  # in this example it cant be later than 2025-12-30 because of the options data availability
V_0              = 1.5  # Upper bound for the dispersion IV constraint this time defined as IVstock / IVindex <= V_0 expected values gtw 1.2 and 2
K                = 15    # Number of stocks to include in the portfolio     

In [3]:
DATE_MIN = "2023-01-03"
DATE_MAX = "2025-12-15"
V_0_MIN  = 0.6

if date_from < DATE_MIN:
    raise ValueError(f"date_from={date_from!r} is before the earliest available date {DATE_MIN!r}.")
if date_to > DATE_MAX:
    raise ValueError(f"date_to={date_to!r} is after the latest available date {DATE_MAX!r}.")
if date_from >= date_to:
    raise ValueError(f"date_from={date_from!r} must be strictly before date_to={date_to!r}.")
if V_0 < V_0_MIN:
    raise ValueError(f"V_0={V_0} is below the minimum value {V_0_MIN} — the problem may have no feasible solution.")

print("Configuration validated successfully.")


Configuration validated successfully.


---

## Step 2 — Load Price Data

We load **daily adjusted close prices** for each NDX constituent stock and for the index itself.

Each stock file contains OHLCV data adjusted for splits and dividends. The prices are assembled into a single `DataFrame`:

$$
P \in \mathbb{R}^{T \times (N+1)}, \qquad P_{t,i} = \text{closing price of asset } i \text{ on day } t
$$

where $T$ is the number of trading days and $N$ is the number of constituent stocks (the last column is the NDX index itself).

> **Note on NaNs:** Stocks that were not yet listed or had no trading activity on a given date appear as `NaN`. These are handled automatically when estimating the cost matrix in the next step.

In [4]:

PRICE_COLS = ["Date", "Open", "High", "Low", "Close", "Volume"]
NDX_COLS   = ["Date", "Open", "High", "Low", "Close"]

all_close_prices = {}

for filepath in glob.glob(os.path.join(STOCK_PRICES_DIR, "*_full_1day_adjsplitdiv.txt")):
    ticker = os.path.basename(filepath).replace("_full_1day_adjsplitdiv.txt", "")
    df = pd.read_csv(filepath, header=None, names=PRICE_COLS, parse_dates=["Date"], index_col="Date")
    all_close_prices[ticker] = df["Close"]

ndx_df = pd.read_csv(NDX_PRICE_FILE, header=None, names=NDX_COLS, parse_dates=["Date"], index_col="Date")
all_close_prices[BENCHMARK_COLUMN] = ndx_df["Close"]

assets_index_prices = pd.DataFrame(all_close_prices)

print("\nAsset Index Prices (first 5 rows):")
print(assets_index_prices.head())


Asset Index Prices (first 5 rows):
              AAPL  ABNB     ADBE      ADI  ADP    ADSK      AEP     AMAT  \
Date                                                                        
2000-01-04  0.7788   NaN  15.0156  27.8177  NaN  7.9375  13.5253  25.3969   
2000-01-05  0.7902   NaN  15.3125  28.2238  NaN  7.4375  14.0302  24.4465   
2000-01-06  0.7218   NaN  15.4375  27.4522  NaN  6.9844  14.1099  24.4861   
2000-01-07  0.7560   NaN  16.1875  28.2238  NaN  7.8125  14.2959  24.6973   
2000-01-10  0.7427   NaN  16.8125  30.6807  NaN  7.7188  14.2428  26.6905   

               AMD     AMGN  ...  TTD  TTWO      TXN  VRSK     VRTX  WBD  \
Date                         ...                                           
2000-01-04  14.625  40.7687  ...  NaN   NaN  31.5075   NaN  17.2812  NaN   
2000-01-05  15.000  42.1715  ...  NaN   NaN  30.7478   NaN  17.0000  NaN   
2000-01-06  16.000  42.8729  ...  NaN   NaN  29.9081   NaN  16.7500  NaN   
2000-01-07  16.250  47.6950  ...  NaN   NaN 

---

## Step 3 — Estimate the quadratic distance



In [5]:
cost_matrix, benchmark_cost_vector, tickers = iq.tools.quadratic_utility.compute_quadratic_utility(
    assets_index_prices=assets_index_prices,
    benchmark_column=BENCHMARK_COLUMN, 
    str_cost_function = "quadratic_distance",
    date_from=date_from,
    date_to=date_to,
    EMWA_halflife=252,
    maximum_missing_data_ratio_allowed=0.25,
    l2_regularization=0.01
)
cost_matrix, benchmark_cost_vector, tickers = iq.tools.quadratic_utility.compute_covariance_matrix(
    assets_index_prices=assets_index_prices,
    benchmark_column=BENCHMARK_COLUMN, 
    date_from=date_from,
    date_to=date_to,
    EMWA_halflife=252,
    maximum_missing_data_ratio_allowed=0.25,
    l2_regularization=0.01
)

print("\nCost Matrix shape:", cost_matrix.shape)
print("\nCost Vector:")
print(benchmark_cost_vector)
print("\nTickers:")
print(tickers)


Cost Matrix shape: (101, 101)

Cost Vector:
[ 1.68960381  1.84016102  1.46107426  2.18232028  0.65435736  1.43557021
 -0.10256848  2.6403183   3.11801463  0.49611127  2.02761397  3.35992893
  3.91142267  2.43077269  3.09527079  1.4922213   0.37766125  0.63689964
  1.32586093  1.26233915  0.26578702  2.19761665  1.38083944  2.34620673
  0.91224924  0.74315459  0.80385453  0.91200535  2.45146285  1.08759925
  1.08535785  0.82342745  0.84604831  1.05754114  1.89201698  2.10163251
  1.25805645  0.53122942 -0.13351956  1.20175244  0.76291628  1.56283653
  1.512211    2.42983059  0.33640489  1.6415018   1.63523494  0.89404964
  1.36563079  2.25912917  1.41334396  1.67706048  0.13373689  0.15562711
  2.73159157  0.70677817  2.91021474  1.54441781  1.40030877  2.89080922
  0.02086572  1.30610259  2.13909602  0.39803743  3.72114207  1.44643885
  3.34744466  3.1092722   1.45532153  3.2078073   2.54725979  1.4618743
  2.87003544  0.21255486  1.75576904  0.6754604   1.02509527  1.17140105
  0.036

---

## Step 4 — Build the IV Constraint Matrix

This step is what differentiates a *dispersion trade* from plain index tracking.

### Implied Volatility Ratio

For each stock $i$ and each trading day $t$, we compute the **IV ratio**:

$$
R_{t,i} = \frac{\sigma_i^{\text{call}}(t)}{\sigma_{\text{index}}^{\text{call}}(t)}
$$

where $\sigma_i^{\text{call}}(t)$ is the mid implied volatility of the **front-month, ATM call option** on stock $i$ at date $t$.

### Portfolio-Level IV Constraint

We require that the portfolio's *aggregate* IV exposure relative to the index stays bounded on every date:

$$
\sum_{i=1}^N R_{t,i}\, w_i \leq V_0 \qquad \forall\, t \in \mathcal{T}
$$

Stacking all dates into matrix form:

$$
\mathbf{R}\,\mathbf{w} \leq V_0\,\mathbf{1}, \qquad \mathbf{R} \in \mathbb{R}^{|\mathcal{T}| \times N}
$$

This is passed to the solver as a standard linear inequality constraint.

**IV extraction rules applied per ticker, per date:**

- Only **call** options are used (type `"c"`)
- We use the **nearest available expiry** (front month) to get a liquid, representative IV
- Mid IV is computed as:

$$\text{Mid IV} = \frac{\text{Bid IV} + \text{Ask IV}}{2}$$

- The **same rules** apply to NDX index options to obtain $\sigma_{\text{index}}^{\text{call}}(t)$
- Dates where any ticker is missing IV data are dropped from $\mathbf{R}$ to keep the solver JSON-safe

In [6]:
def get_daily_front_expiry_call_iv(df_opt, date_from, date_to):
    """Return a Series (index=Trade Date) of mid IV for the nearest-expiry call."""
    df = df_opt[
        (df_opt["Trade Date"] >= date_from) &
        (df_opt["Trade Date"] <= date_to) &
        (df_opt["Call/Put"] == "c")
    ].copy()

    # Keep only the nearest expiry for each trade date
    min_expiry = df.groupby("Trade Date")["Expiry Date"].transform("min")
    df = df[df["Expiry Date"] == min_expiry]

    df["mid_iv"] = (df["Bid Implied Volatility"] + df["Ask Implied Volatility"]) / 2
    return df.set_index("Trade Date")["mid_iv"]


daily_ivs = {}

for ticker in tickers:
    filepath = os.path.join(OPTIONS_DATA_DIR, f"{ticker}_filtered.csv")
    if not os.path.exists(filepath):
        daily_ivs[ticker] = pd.Series(dtype=float)
        continue
    df_opt = pd.read_csv(filepath, parse_dates=["Trade Date", "Expiry Date"])
    daily_ivs[ticker] = get_daily_front_expiry_call_iv(df_opt, date_from, date_to)

# DataFrame: index=dates, columns=tickers — daily front-expiry call IV per stock
iv_df = pd.DataFrame(daily_ivs).sort_index()

# NDX index IV: front-expiry call IV from NDX_filtered.csv
df_ndx_opt = pd.read_csv(NDX_IV_FILE, parse_dates=["Trade Date", "Expiry Date"])
iv_index_series = get_daily_front_expiry_call_iv(df_ndx_opt, date_from, date_to)

# Divide each stock's daily IV by the index IV for that day
# Result shape: (num_dates, num_tickers)
linear_constraints_matrix = (iv_df.div(iv_index_series, axis=0)).to_numpy()

# Drop rows where any ticker is missing IV data (NaN) — JSON does not accept NaN
valid_rows = ~np.isnan(linear_constraints_matrix).any(axis=1)
linear_constraints_matrix = linear_constraints_matrix[valid_rows]

num_dates = linear_constraints_matrix.shape[0]
lower_bounds_vector = np.zeros(num_dates)
upper_bounds_vector = np.full(num_dates, V_0)

print(f"\nIV constraint matrix shape: {linear_constraints_matrix.shape}")
print(f"Date range in IV data: {iv_df.index[0].date()} to {iv_df.index[-1].date()}")



IV constraint matrix shape: (518, 101)
Date range in IV data: 2023-02-03 to 2025-10-15


---

## Step 5 — Solve the Index-Tracking Optimization

We now solve the full **mixed-integer quadratic program (MIQP)** that selects exactly $K$ stocks and finds the optimal weights.

### Problem Formulation

$$
\min_{\mathbf{w},\, \mathbf{z}} \quad \mathbf{w}^\top \Sigma\, \mathbf{w} - 2\,\mathbf{c}^\top \mathbf{w}
$$

$$
\text{subject to} \quad \sum_{i=1}^N w_i = 1
$$

$$
w_{\min} \leq w_i \leq w_{\max} \cdot z_i, \quad z_i \in \{0, 1\}
$$

$$
\sum_{i=1}^N z_i = K \quad \text{(cardinality constraint)}
$$

$$
\mathbf{R}\,\mathbf{w} \leq V_0\ \quad \text{(IV dispersion constraints)}
$$

The binary variables $z_i$ select which $K$ stocks enter the portfolio; the continuous variables $w_i$ determine their weights. The solver returns the **optimal weights** $w_i^*$ for the selected stocks.

In [7]:
portfolio, status, error_description = iq.finance.index_tracking.solve_index_tracking(
    description="Index Tracking - Dispersion Trade",
    assets_utility_matrix=cost_matrix,
    assets_to_benchmark_utility_vector=benchmark_cost_vector,
    portfolio_size=K,
    asset_names=tickers,
    minimum_weight=0.01,
    linear_constraints_matrix=linear_constraints_matrix,
    lower_bounds_vector=lower_bounds_vector,
    upper_bounds_vector=upper_bounds_vector,
    maximum_weight=0.55
)

print("\nOptimal Portfolio Weights:")
portfolio = {asset: round(100 * weight, 4) for asset, weight in portfolio.items()}
print(portfolio)

if status == "Ok":
    print("✅ No errors encountered.")
elif error_description:
    print(f"❌ Error: {error_description}")


solution found

Optimal Portfolio Weights:
{'AAPL': 1.0, 'ABNB': 0.0, 'ADBE': 0.0, 'ADI': 0.0, 'ADP': 0.0, 'ADSK': 1.0, 'AEP': 0.0, 'AMAT': 0.0, 'AMD': 0.0, 'AMGN': 0.0, 'AMZN': 0.0, 'APP': 0.0, 'ARM': 0.0, 'ASML': 0.0, 'AVGO': 0.0, 'AXON': 0.0, 'AZN': 0.0, 'BIIB': 0.0, 'BKNG': 0.0, 'BKR': 0.0, 'CCEP': 1.0, 'CDNS': 0.0, 'CDW': 0.0, 'CEG': 0.0, 'CHTR': 0.0, 'CMCSA': 0.0, 'COST': 7.3802, 'CPRT': 0.0, 'CRWD': 0.0, 'CSCO': 0.0, 'CSGP': 0.0, 'CSX': 0.0, 'CTAS': 0.0, 'CTSH': 0.0, 'DASH': 0.0, 'DDOG': 0.0, 'DXCM': 0.0, 'EA': 0.0, 'EXC': 0.0, 'FANG': 0.0, 'FAST': 1.0, 'FTNT': 0.0, 'GEHC': 0.0, 'GFS': 0.0, 'GILD': 0.0, 'GOOGL': 0.0, 'GOOG': 1.0, 'HON': 19.6394, 'IDXX': 0.0, 'INTC': 0.0, 'INTU': 1.3315, 'ISRG': 0.0, 'KDP': 0.0, 'KHC': 0.0, 'KLAC': 0.0, 'LIN': 22.3354, 'LRCX': 0.0, 'LULU': 0.0, 'MAR': 0.0, 'MCHP': 0.0, 'MDLZ': 0.0, 'MELI': 0.0, 'META': 0.0, 'MNST': 0.0, 'MRVL': 0.0, 'MSFT': 10.6812, 'MSTR': 0.0, 'MU': 0.0, 'NFLX': 0.9999, 'NVDA': 4.7551, 'NXPI': 0.0, 'ODFL': 0.0, 'ON': 0.0, 'ORLY

---

## Results & Next Steps

The portfolio weights printed above are the **optimal allocations (in %)** for each of the $K$ selected stocks.

### Interpreting the Output

| Field | Meaning |
|---|---|
| `portfolio` | `{ticker: weight%}` — only the $K$ selected stocks appear |
| `status` | `"Ok"` if the solver found a feasible, optimal solution |
| `error_description` | Non-empty string if the problem was infeasible or an error occurred |

### Why Might the Problem be Infeasible?

- **$V_0$ is too low** → the IV constraint is too tight for any $K$-stock portfolio to satisfy; try increasing `V_0`
- **Sparse options data** → too many dates have missing IVs after filtering; try a narrower or more recent date range
- **Large $K$** → more stocks can make it harder to satisfy both the cardinality and IV constraints simultaneously

### What to Try Next

- Increase `V_0` if the problem is infeasible
- Change `date_from` / `date_to` to a different window
- Vary `K` to explore the cardinality–tracking-error frontier
- Backtest the returned portfolio over a hold-out period to evaluate out-of-sample tracking quality